《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 가설검정

앞 장들에서는 집단 사이의 차이와 변수 사이의 상관을 살펴보았습니다. 사람과 펭귄의 자료에서 모두 이런 결과를 보았습니다. 이런 결과는 표본에서 나타났다는 뜻에서 **관측된 효과**라고 부릅니다. 보통 직접 볼 수 없는 모집단의 실제 효과와 구별하는 말입니다. 표본에서 효과를 발견하면, 모집단에도 있는 현상인지 표본의 우연한 변동인지 생각해야 합니다.

이 질문을 다루는 방법으로 피셔의 귀무가설 검정, 네이만–피어슨의 의사결정 이론, 베이즈 가설검정 등이 있습니다. 여기서는 실제 분석에서 흔히 쓰는, 이 접근들이 섞인 틀을 설명하겠습니다.

[원문 노트북을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/30ecbdcf05723f1190cae2aeaff5848f59f8ca26/nb/chap09.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## 동전 돌리기

David MacKay의 《Information Theory, Inference, and Learning Algorithms》에 나온 예로 시작하겠습니다.

2002년 유로 동전이 도입되었을 때, 한 동전 애호가가 벨기에의 1유로 동전을 세워 250번 돌렸습니다. 앞면은 140번, 뒷면은 110번 나왔습니다. 공정한 동전이라면 앞면의 기댓값은 125번이므로, 앞면이 더 자주 나오는 동전인지 궁금해집니다.

그렇다고 공정한 동전에서 매번 정확히 125번의 앞면이 나와야 하는 것은 아닙니다. 이 차이가 우연한 변동으로 설명될 만한지 가설검정으로 알아보겠습니다. 다음 함수는 공정한 동전에서 기대하는 앞면 횟수와 관측 횟수 사이의 절대편차를 구합니다.

In [4]:
n = 250
p = 0.5


def abs_deviation(heads):
    expected = n * p
    return np.abs(heads - expected)

관측 자료의 절대편차는 15입니다.

In [5]:
heads = 140
tails = 110

observed_stat = abs_deviation(heads)
observed_stat

동전이 공정하다면 앞면과 뒷면을 같은 확률로 생성해 실험을 흉내 낼 수 있습니다. `H`와 `T`로 이루어진 난수 시퀀스를 만들고 `H`의 개수를 세겠습니다.

In [6]:
def simulate_flips():
    flips = np.random.choice(["H", "T"], size=n)
    heads = np.sum(flips == "H")
    return heads

함수를 호출할 때마다 모의실험 한 번의 결과가 나옵니다.

In [7]:
np.random.seed(1)

In [8]:
simulate_flips()

다음 반복문은 실험을 여러 번 시행하고, 각 실험의 절대편차를 리스트에 모읍니다.

In [9]:
simulated_stats = [abs_deviation(simulate_flips()) for i in range(10001)]

결과는 동전이 공정하다는 가정 아래에서 편차가 어떻게 분포하는지 보여 주는 표본입니다. 그래프로 그려 보겠습니다.

In [10]:
from empiricaldist import Pmf

pmf_effects = Pmf.from_seq(simulated_stats)
pmf_effects.bar()

decorate(xlabel="Absolute deviation", ylabel="PMF")

0에 가까운 편차가 흔하고, 10을 넘는 편차는 드뭅니다. 실제 편차 15만큼 큰 값도 드물지만 불가능하지는 않습니다. 원문의 모의실험에서는 약 7.1%가 15 이상이었습니다.

In [11]:
(np.array(simulated_stats) >= 15).mean() * 100

즉 동전이 공정해도, 이 정도 이상의 편차가 원문의 모의실험에서는 약 7.1% 나타났습니다. 이 크기의 효과는 흔하지 않지만 공정한 동전에서도 생길 수 있으므로, 이 실험만으로 공정할 가능성을 배제하기는 어렵습니다.

이 예에 가설검정의 기본 요소가 들어 있습니다.

- 관측: 250번 중 앞면이 140번 나왔습니다. 검토할 가설은 앞면 확률이 50%와 다르다는 것입니다.
- **검정통계량**: 자료가 귀무가설과 얼마나 어긋나는지 수치로 요약합니다. 여기서는 앞면 횟수와 기대 횟수 차이의 절댓값입니다.
- **귀무가설**: 관측된 효과가 우연한 변동으로 생긴다고 가정하는 모형입니다. 여기서는 동전이 공정하다는 가정입니다.
- **p값**: 귀무가설이 참일 때, 관측값만큼 또는 그보다 극단적인 검정통계량이 나올 확률입니다. 여기서는 편차가 15 이상일 확률입니다.

마지막으로 결과를 해석합니다. p값이 작으면 그 귀무모형에서 관측 결과가 드물다는 뜻입니다. 크면 그 모형의 우연한 변동으로 설명할 여지가 있습니다. 중간에 있다면 판단도 조심스러워야 합니다. 어떤 검정이든 통계량과 귀무가설, 그 가정 아래의 p값을 구별해야 합니다.

> 번역자 주: p값은 자료를 본 뒤 계산한 ‘귀무가설이 참일 확률’이나 ‘결과가 우연일 확률’이 아닙니다. 큰 p값도 귀무가설이 옳다는 증명은 아닙니다. 여기서 확률을 계산할 때는 앞면 확률 0.5를 고정해 둡니다.

## 평균 차이 검정하기

NSFG 자료에서 첫아이의 평균 임신 기간은 다른 아이보다 조금 길었습니다. 이 차이가 우연한 변동으로도 나타날 수 있는지 알아보겠습니다.

1장과 같은 방식으로 읽고 정제한 NSFG 자료를 사용합니다.

In [12]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/nsfg.py")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [13]:
try:
    import statadict
except ImportError:
    %pip install statadict

`get_nsfg_groups`는 자료를 읽고 생아 출산을 고른 뒤, 첫아이와 다른 아이로 나눕니다.

In [14]:
from nsfg import get_nsfg_groups

live, firsts, others = get_nsfg_groups()

두 집단에서 주 단위 임신 기간을 선택합니다.

In [15]:
data = firsts["prglngth"].values, others["prglngth"].values

다음 함수는 두 시퀀스를 튜플로 받아 평균 차이의 절댓값을 반환합니다.

In [16]:
def abs_diff_means(data):
    group1, group2 = data
    diff = np.mean(group1) - np.mean(group2)
    return np.abs(diff)

관측된 평균 임신 기간의 차이는 약 0.078주입니다.

In [17]:
observed_diff = abs_diff_means(data)
observed_diff

검토할 가설은 첫아이와 다른 아이의 임신 기간이 다르다는 것입니다. 귀무가설은 두 집단의 임신 기간 분포가 같고, 관측된 차이는 우연한 표집 변동이라는 것입니다.

분포가 같다고 가정하면 두 집단을 하나로 합칠 수 있습니다. 합친 자료의 순서를 무작위로 섞은 뒤, 원래 표본 크기와 같은 두 부분으로 나눕니다. 이렇게 집단 이름을 바꾸어 붙이는 과정으로 귀무모형의 실험을 만듭니다.

In [18]:
def simulate_groups(data):
    group1, group2 = data
    n, m = len(group1), len(group2)

    pool = np.hstack(data)
    np.random.shuffle(pool)
    return pool[:n], pool[-m:]

함수는 두 시퀀스의 튜플을 반환하므로 `abs_diff_means`에 바로 전달할 수 있습니다.

In [19]:
abs_diff_means(simulate_groups(data))

다음 반복문은 가상 자료를 여러 번 만들고 각각의 평균 차이를 계산합니다.

In [20]:
simulated_diffs = [abs_diff_means(simulate_groups(data)) for i in range(1001)]

시뮬레이션 결과를 그릴 때는 다음 함수로 밀도 곡선을 근사하겠습니다. 여기서 `Pmf`는 평가점과 밀도를 담는 용도로 쓰며, 정규화된 확률질량함수를 뜻하지는 않습니다.

In [21]:
from scipy.stats import gaussian_kde
from empiricaldist import Pmf


def make_pmf(sample, low, high):
    kde = gaussian_kde(sample)
    qs = np.linspace(low, high, 201)
    ps = kde(qs)
    return Pmf(ps, qs)

다음 함수는 관측값 오른쪽이나 왼쪽의 곡선 아래를 채웁니다.

In [22]:
from thinkstats import underride


def fill_tail(pmf, observed, side, **options):
    """Fill the area under a PMF, right or left of an observed value."""
    options = underride(options, alpha=0.3)

    if side == "right":
        condition = pmf.qs >= observed
    elif side == "left":
        condition = pmf.qs <= observed

    series = pmf[condition]
    plt.fill_between(series.index, 0, series, **options)

다음은 귀무모형에서 얻은 평균 차이의 분포입니다. 관측된 차이 이상인 오른쪽 꼬리 부분이 p값에 대응합니다.

In [23]:
pmf = make_pmf(simulated_diffs, 0, 0.2)
pmf.plot()
fill_tail(pmf, observed_diff, "right")
decorate(xlabel="Absolute difference in means (weeks)", ylabel="Density")

p값은 관측값 이상인 모의실험 결과가 차지하는 비율로 계산하겠습니다.

In [24]:
def compute_p_value(simulated, observed):
    """Fraction of simulated values as big or bigger than the observed value."""
    return (np.asarray(simulated) >= observed).mean()

원문의 p값은 약 18%입니다. 같은 분포에서 나온 두 집단에서도 0.078주 이상의 평균 차이가 생길 만하다는 뜻입니다.

In [25]:
compute_p_value(simulated_diffs, observed_diff)

이 결과만으로 첫아이의 임신 기간이 일반적으로 더 길다고 확신하기는 어렵습니다. 이 표본의 차이는 우연한 변동일 수도 있습니다.

여기서도 검정통계량, 귀무가설, 귀무가설을 구현한 모형이 등장합니다. 통계량은 평균 차이의 절댓값입니다. 귀무가설은 임신 기간의 분포가 같다는 것입니다. 두 집단을 합치고 섞어 같은 크기로 다시 나눈 과정은 **순열**(permutation)이라고 합니다.

이 계산 틀은 통계량만 바꾸면 다른 질문에도 사용할 수 있습니다.

> 번역자 주: 집단을 자유롭게 섞으려면 귀무가설 아래에서 관측들이 교환 가능해야 합니다. ‘평균만 같다’는 가정은 ‘분포가 같다’보다 약합니다. 또 같은 산모의 반복 임신과 복합표본 설계는 이 간단한 실습에서 별도로 반영하지 않습니다. 밀도 곡선은 모양을 보여 주는 근사이고, p값 계산에는 원래 시뮬레이션 값들을 씁니다.

## 다른 검정통계량

첫아이의 임신 기간이 더 들쭉날쭉한지도 궁금할 수 있습니다. 이번에는 두 집단의 표준편차 차이의 절댓값을 검정통계량으로 사용하겠습니다.

In [26]:
def abs_diff_stds(data):
    group1, group2 = data
    diff = np.std(group1) - np.std(group2)
    return np.abs(diff)

NSFG 자료에서 두 집단 표준편차의 차이는 약 0.18주입니다.

In [27]:
observed_diff = abs_diff_stds(data)
observed_diff

이 차이를 검토할 때도 같은 순열 방법을 사용합니다. 귀무모형에서 여러 번 자료를 만들고 표준편차 차이를 구합니다.

In [28]:
simulated_diffs = [abs_diff_stds(simulate_groups(data)) for i in range(1001)]

다음은 모의실험 결과의 분포입니다. 관측된 차이 이상의 부분을 확인합니다.

In [29]:
pmf = make_pmf(simulated_diffs, 0, 0.5)
pmf.plot()
fill_tail(pmf, observed_diff, "right")
decorate(xlabel="Absolute difference in std (weeks)", ylabel="Density")

그 부분의 확률은 관측값 이상인 시뮬레이션 결과의 비율로 추정할 수 있습니다.

In [30]:
compute_p_value(simulated_diffs, observed_diff)

원문의 p값은 약 0.17입니다. 두 집단의 분포가 같아도 이 정도 차이가 생길 수 있으므로, 첫아이의 임신 기간이 일반적으로 더 넓게 퍼져 있다고 단정하기 어렵습니다.

## 상관 검정하기

같은 틀로 상관도 검정할 수 있습니다. NSFG 자료에서는 산모 나이와 출생 체중 사이에 양의 상관이 있습니다. 나이가 많은 산모의 아기가 평균적으로 조금 무겁습니다. 이 관계도 우연히 나타난 것일까요?

생아 출산 중 나이와 출생 체중이 모두 알려진 사례를 고릅니다.

In [31]:
valid = live.dropna(subset=["agepreg", "totalwgt_lb"])
valid.shape

필요한 두 열을 선택합니다.

In [32]:
ages = valid["agepreg"]
birthweights = valid["totalwgt_lb"]

다음 함수는 두 시퀀스를 받아 상관계수의 절댓값을 구합니다. 양과 음의 상관을 모두 극단적인 결과로 취급합니다.

In [33]:
def abs_correlation(data):
    xs, ys = data
    corr = np.corrcoef(xs, ys)[0, 1]
    return np.abs(corr)

NSFG 자료의 상관은 약 0.07입니다.

In [34]:
data = ages, birthweights
observed_corr = abs_correlation(data)
observed_corr

귀무모형에서는 나이와 체중의 관계를 끊어 놓겠습니다. 나이의 값들은 그대로 두되 순서를 무작위로 섞으면, 두 변수 각각의 분포는 유지하면서 원래 짝은 사라집니다.

다음 함수는 `xs`만 복사해 섞고 원래 `ys`와 묶어 반환합니다. `ys`를 섞거나 양쪽을 각각 섞어도 같은 목적을 달성할 수 있습니다.

> 번역자 주: 이 순열 모형은 두 변수의 독립을 가정합니다. 피어슨 상관이 0이라는 조건만으로 독립이 보장되지는 않습니다.

In [35]:
def permute(data):
    xs, ys = data
    new_xs = xs.values.copy()
    np.random.shuffle(new_xs)
    return new_xs, ys

짝을 섞은 자료의 상관은 보통 0에 가깝습니다.

In [36]:
abs_correlation(permute(data))

다음 반복문은 순열 자료를 여러 번 만들고 각각의 상관을 구합니다.

In [37]:
simulated_corrs = [abs_correlation(permute(data)) for i in range(1001)]

다음 분포와 관측 상관을 나타내는 수직 점선을 비교해 보세요.

In [38]:
pmf = make_pmf(simulated_corrs, 0, 0.07)
pmf.plot()
plt.axvline(observed_corr, color="gray", ls=":")
decorate(xlabel="Absolute value of correlation", ylabel="Density")

관측 상관은 귀무분포의 아주 먼 꼬리에 있습니다. 원문에서는 시뮬레이션 중 관측값 이상인 상관이 한 번도 없어 계산 결과가 0입니다.

In [39]:
compute_p_value(simulated_corrs, observed_corr)

계산 결과가 0이라고 해서 실제 꼬리 확률이 정확히 0인 것은 아닙니다. 드문 결과를 이번 시뮬레이션에서 보지 못했을 뿐입니다.

관습적으로 p값이 0.05보다 작으면 **통계적으로 유의하다**고 부릅니다. 하지만 기준 자체가 임의적이고, ‘유의하다’는 말이 실질적으로 중요하다는 뜻처럼 들려 오래전부터 문제가 지적되어 왔습니다. 산모 나이와 출생 체중의 상관이 좋은 예입니다. p값은 작지만 관계는 약합니다.

p값을 정성적으로 읽는 방법도 있습니다. 큰 값이면 귀무모형의 변동으로 자료를 설명할 여지가 있습니다. 작은 값이면 그 모형에서 관측 결과가 드뭅니다. 그렇더라도 대표성 없는 표본이나 측정 오류 같은 다른 원인은 남습니다.

> 번역자 주: 1001번 중 0번 관측했다는 사실만으로 $p<0.001$을 보장할 수는 없습니다. 독립적인 모의실험의 이항 모형에서 이때 꼬리 확률의 단측 95% 상한은 약 0.003입니다. 아래 브라우저 순열 실습은 0을 출력하지 않도록 `(극단적 결과 수+1)/(반복 수+1)`을 사용합니다. 검정통계량과 분석 방법도 결과를 보기 전에 정하는 것이 좋습니다.

## 비율 검정하기

마지막으로 통계량을 고르는 데 생각이 필요한 예를 보겠습니다. 카지노에서 손님이 특정 눈이 잘 나오도록 조작한 주사위를 쓴다고 의심한다고 하겠습니다. 주사위를 확보한 뒤 60번 던져 1부터 6까지의 도수를 기록했습니다. 결과를 `Hist`로 나타내겠습니다.

In [40]:
from empiricaldist import Hist

qs = np.arange(1, 7)
freqs = [8, 9, 19, 5, 8, 11]
observed = Hist(freqs, qs)
observed.index.name = "outcome"
observed

공정한 주사위라면 각 눈의 기대 도수는 10입니다. 관측에서는 3이 많고 4가 적습니다. 이 차이도 우연히 생길 수 있는지 알아보기 위해 기대 도수를 만듭니다.

In [41]:
num_rolls = observed.sum()
outcomes = observed.qs
expected = Hist(num_rolls / 6, outcomes)

다음 함수는 관측 도수와 기대 도수의 절대편차를 모두 더합니다.

In [42]:
def total_abs_deviation(observed):
    return np.sum(np.abs(observed - expected))

이 자료에서 절대편차의 합은 20입니다.

In [43]:
observed_dev = total_abs_deviation(observed)
observed_dev

다음 함수는 공정한 주사위를 관측 자료와 같은 횟수만큼 던지고, 가상 도수를 `Hist`로 반환합니다.

In [44]:
def simulate_dice(observed):
    num_rolls = np.sum(observed)
    rolls = np.random.choice(observed.qs, num_rolls, replace=True)
    hist = Hist.from_seq(rolls)
    return hist

다음 반복문은 여러 실험에서 절대편차의 합을 구합니다.

In [45]:
simulated_devs = [total_abs_deviation(simulate_dice(observed)) for i in range(1001)]

다음은 귀무가설 아래의 검정통계량 분포입니다. 기대보다 많이 나온 횟수의 합과 적게 나온 횟수의 합이 같으므로, 이 예에서 절대편차의 합은 항상 짝수입니다.

In [46]:
pmf_devs = Pmf.from_seq(simulated_devs)
pmf_devs.bar()

decorate(xlabel="Total absolute deviation", ylabel="PMF")

편차의 합 20은 특별히 드물지 않습니다. 원문의 p값은 약 13%로, 이 결과만으로 주사위가 조작되었다고 확신하기 어렵습니다.

In [47]:
compute_p_value(simulated_devs, observed_dev)

하지만 절대편차의 합만이 가능한 통계량은 아닙니다. 이 문제에는 보통 다음 **카이제곱 통계량**을 사용합니다.

In [48]:
def chi_squared_stat(observed):
    diffs = (observed - expected) ** 2
    return np.sum(diffs / expected)

편차를 제곱하면 큰 편차에 더 큰 가중치가 붙습니다. 기대 도수로 나누면 기대 크기에 맞춰 편차를 조정할 수 있습니다. 이 예에서는 기대 도수가 모두 같으므로, 그 나눗셈은 통계량의 공통 배율만 바꾸고 결과의 순서는 바꾸지 않습니다.

In [49]:
observed_chi2 = chi_squared_stat(observed)
observed_chi2

관측된 카이제곱 통계량은 11.6입니다. 숫자만으로 해석하기보다, 공정한 주사위의 모의실험 결과와 비교하겠습니다.

In [50]:
simulated_chi2 = [chi_squared_stat(simulate_dice(observed)) for i in range(1001)]

다음 분포의 오른쪽 꼬리는 관측 카이제곱 통계량 이상인 결과들입니다.

In [51]:
pmf = make_pmf(simulated_chi2, 0, 20)
pmf.plot()
fill_tail(pmf, observed_chi2, "right")
decorate(xlabel="Chi-Squared statistic", ylabel="Density")

이 꼬리 부분의 확률이 p값입니다.

In [52]:
compute_p_value(simulated_chi2, observed_chi2)

카이제곱 통계량을 쓴 원문의 p값은 약 0.04로, 절대편차로 구한 약 0.13보다 작습니다. 5% 기준만 적용하면 유의하다고 하겠지만, 두 검정을 함께 보면 결과가 결정적이라고 말하기 어렵습니다. 저라면 조작 가능성을 배제하지 않겠지만, 이 자료만으로 손님에게 유죄를 선고하지도 않겠습니다.

p값은 검정통계량과 귀무모형의 선택에 달려 있습니다. 때로는 그 선택이 ‘유의하다’는 판정까지 바꿉니다.

> 번역자 주: 여러 통계량을 시도한 뒤 가장 작은 p값만 고르면 분석 선택의 효과가 섞입니다. 브라우저 코드는 눈이 한 번도 나오지 않은 경우에도 도수 배열의 여섯 칸을 모두 남겨, 기대 도수와 정확히 비교합니다.

## 용어 정리

- **가설검정**: 관측된 효과가 귀무모형의 표집 변동으로 설명될 만한지 살피는 방법들.
- **검정통계량**: 자료가 귀무가설과 어긋나는 정도를 요약해, 꼬리 확률 계산에 사용하는 값.
- **귀무가설**: 검정의 비교 기준으로 삼는 모집단에 대한 가정. 이 장에서는 동전이 공정하다거나, 두 집단의 분포가 같다거나, 두 변수가 독립이라는 가정입니다.
- **순열**: 자료의 순서를 무작위로 섞어 귀무모형을 구현하는 방법.
- **p값**: 귀무가설 아래에서 관측값 이상으로 극단적인 통계량이 나올 확률.
- **통계적 유의성**: p값이 미리 정한 기준보다 작은 상태. 표본이 크면 실질적으로 작은 효과도 유의할 수 있습니다.

## 연습문제

### 연습문제 9.1

[8장의 펭귄 자료](/posts/statistics/think-stats-08#표집분포)로 가설검정을 해 보세요. 자료를 불러오는 방법은 이 장의 노트북에 있습니다.

자료는 Allison Horst가 만든 저장소에서 가져옵니다.

Horst, Hill, Gorman(2020), *palmerpenguins*, DOI: [10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218). 자료를 수집한 연구는 Gorman, Williams, Fraser(2014), 「Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins」, *PLOS ONE* 9(3): e90081, DOI: [10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081)입니다.

In [53]:
download(
    "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/c19a904462482430170bfe2c718775ddb7dbb885/inst/extdata/penguins_raw.csv"
)

자료를 읽고 턱끈펭귄을 고릅니다.

In [54]:
penguins = pd.read_csv("penguins_raw.csv").dropna(subset=["Body Mass (g)"])
chinstrap = penguins.query('Species.str.startswith("Chinstrap")')
chinstrap.shape

수컷과 암컷의 체중을 각각 킬로그램 단위로 꺼냅니다.

In [55]:
male = chinstrap.query("Sex == 'MALE'")
weights_male = male["Body Mass (g)"] / 1000
weights_male.mean()

In [56]:
female = chinstrap.query("Sex == 'FEMALE'")
weights_female = female["Body Mass (g)"] / 1000
weights_female.mean()

두 집단의 체중 분포가 같다는 귀무가설 아래에서 `abs_diff_means`와 `simulate_groups`로 모의실험을 여러 번 하세요. 관측 평균 차이와 비교해 p값을 구하세요. 이 차이가 우연한 변동으로도 나타날 만한가요?

In [57]:
# Solution goes here

In [58]:
# Solution goes here

In [59]:
# Solution goes here

In [60]:
# Solution goes here

In [61]:
# Solution goes here

### 연습문제 9.2

앞 문제의 암컷 펭귄에서 부리의 깊이와 길이를 꺼냅니다. culmen은 부리 위쪽의 능선을 가리킵니다.

In [62]:
data = female["Culmen Depth (mm)"], female["Culmen Length (mm)"]

두 변수의 관측 상관은 약 0.26입니다.

In [63]:
observed_corr = abs_correlation(data)
observed_corr

실제로 두 측정값이 독립이어도 이 정도 상관이 나타날 수 있을까요? `permute`로 짝을 섞고 `abs_correlation`으로 상관의 절댓값을 구하는 실험을 여러 번 하세요. 귀무분포를 그리고 p값을 계산한 뒤 해석하세요.

In [64]:
# Solution goes here

In [65]:
# Solution goes here

In [66]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)